In [8]:
import sys
from pathlib import Path

# Notebook 的工作目录通常是当前文件所在目录，向上查找服务根目录。
service_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "app" / "core" / "config.py").is_file()
    ),
    None,
)
if service_root is None:
    raise RuntimeError("未找到 agent-service 根目录，请从该项目或 notebooks 子目录运行。")

if str(service_root) not in sys.path:
    sys.path.insert(0, str(service_root))


In [11]:
from app.core.config import settings
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from psycopg import AsyncConnection
from psycopg.rows import DictRow, dict_row
from psycopg_pool import AsyncConnectionPool

load_dotenv()
# 创建连接池
checkpoint_pool: AsyncConnectionPool[AsyncConnection[DictRow]] = AsyncConnectionPool(
    conninfo=settings.db.checkpoint_url,
    min_size=1,  # 池最小连接数
    max_size=5,  # 池最大连接数
    kwargs={
        "autocommit": True,  # 自动提交事务
        "prepare_threshold": 0,  # 不做预准备sql
        "row_factory": dict_row,  # 查询结果以dict返回
    },
    open=False,  # 不自动创建连接
)

# 初始化连接
await checkpoint_pool.open()

# 等待连接池就绪
await checkpoint_pool.wait()

# 初始化checkpointer
checkpointer = AsyncPostgresSaver(checkpoint_pool)
# 自动建表（与checkpointer存储有关的表）
await checkpointer.setup()


In [12]:
# 初始化模型
model = init_chat_model(
    "deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}},
)

# 创建Agent
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer,
)

In [13]:
from langchain_core.runnables import RunnableConfig

config: RunnableConfig = {"configurable": {"thread_id": "day06-checkpointer-test"}}

# 第一次，告知自己的信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我叫小明，今年30岁。")]},
    config=config,
)

# 第二次，询问信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我今年多大？")]},
    config=config,
)
print(result["messages"][-1].content)

你今年30岁，小明。😊


In [14]:
# 获取当前会话最新的 state 快照
snapshot = await agent.aget_state(config)
snapshot

StateSnapshot(values={'messages': [HumanMessage(content='我叫小明，今年30岁。', additional_kwargs={}, response_metadata={}, id='422f9857-9ab6-475d-99e2-b85ad33f7fe2'), AIMessage(content='你好，小明！很高兴认识你。30岁是一个很好的年纪，既有一些人生经验，又充满活力和可能性。有什么我可以帮你的吗？无论是聊天、解答问题，还是需要一些建议，我都很乐意帮忙。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 11, 'total_tokens': 58, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 11}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '69ae30c2-1bf8-422d-891f-975dd657e4e4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1d93a-ed91-72f1-b799-656910fc5936-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 47, 't

In [15]:
messages = snapshot.values.get("messages", [])
for message in messages:
    message.pretty_print()

================================ Human Message =================================

我叫小明，今年30岁。
================================== Ai Message ==================================

你好，小明！很高兴认识你。30岁是一个很好的年纪，既有一些人生经验，又充满活力和可能性。有什么我可以帮你的吗？无论是聊天、解答问题，还是需要一些建议，我都很乐意帮忙。
================================ Human Message =================================

我今年多大？
================================== Ai Message ==================================

你今年30岁。你刚才在第一句话里告诉我了：“我叫小明，今年30岁。” 😊
================================ Human Message =================================

我叫小明，今年30岁。
================================== Ai Message ==================================

你好小明！30岁正是好年纪。有什么想聊的或者需要帮忙的吗？😊
================================ Human Message =================================

我今年多大？
================================== Ai Message ==================================

你今年30岁呀，小明。你前面说过“今年30岁”了。😊
================================ Human Message =================================

我叫小明，今年30岁。
=================================

In [16]:
# 删除会话中的全部 state
configurable = config.get("configurable")
if configurable is None:
    raise KeyError("RunnableConfig 中缺少 configurable")

thread_id = configurable.get("thread_id")
if thread_id is None:
    raise KeyError("RunnableConfig 中缺少 thread_id")

await checkpointer.adelete_thread(thread_id)

In [17]:
# 再次读取 state
snapshot = await agent.aget_state(config)
snapshot

StateSnapshot(values={}, next=(), config={'recursion_limit': 9999, 'metadata': {'ls_integration': 'langchain_create_agent'}, 'configurable': {'thread_id': 'day06-checkpointer-test'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())